In [1]:
# Este taller muestra cómo operan los operadores clave–valor antes de aplicarlos a un caso analítico.
# Usaremos dos conductores para que cada entrada y salida pueda leerse durante la demostración.
#
# registros ──► mapper ──► parejas ──► shuffle ──► grupos ──► reducer ──► parejas reducidas

import csv
from pathlib import Path

ROOT = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "data").is_dir() and (path / "submission").is_dir()
)

with (ROOT / "data" / "timesheet.csv").open(encoding="utf-8", newline="") as file:
    timesheet = list(csv.DictReader(file))

with (ROOT / "data" / "drivers.csv").open(encoding="utf-8", newline="") as file:
    drivers = list(csv.DictReader(file))

sample_ids = {"10", "11"}
sample_timesheet = [
    row
    for row in timesheet
    if row["driverId"] in sample_ids and int(row["week"]) <= 2
]
sample_drivers = [row for row in drivers if row["driverId"] in sample_ids]

sample_timesheet

[{'driverId': '10', 'week': '1', 'hours-logged': '70', 'miles-logged': '3300'},
 {'driverId': '10', 'week': '2', 'hours-logged': '70', 'miles-logged': '3300'},
 {'driverId': '11', 'week': '1', 'hours-logged': '50', 'miles-logged': '3000'},
 {'driverId': '11', 'week': '2', 'hours-logged': '83', 'miles-logged': '4000'}]

In [2]:
# map_pairs recibe los datos y una implementación de mapper; no conoce el dominio del caso.

def map_pairs(records, mapper):
    pairs = []

    for record in records:
        pairs.extend(mapper(record))

    return pairs

In [3]:
# Esta implementación del mapper conoce los turnos y produce una pareja por registro.

def map_timesheet_hours(record):
    return [(record["driverId"], float(record["hours-logged"]))]


hour_pairs = map_pairs(sample_timesheet, map_timesheet_hours)

hour_pairs

[('10', 70.0), ('10', 70.0), ('11', 50.0), ('11', 83.0)]

In [4]:
# group_by_key reúne los valores que comparten clave sin saber qué representan.

def group_by_key(pairs):
    grouped_values = {}

    for key, value in pairs:
        grouped_values.setdefault(key, []).append(value)

    return grouped_values

In [5]:
# El shuffle conserva una clave y todos sus valores asociados.

hours_by_driver = group_by_key(hour_pairs)

hours_by_driver

{'10': [70.0, 70.0], '11': [50.0, 83.0]}

In [6]:
# reduce_by_key aplica una implementación de reducer a cada clave y su grupo de valores.

def reduce_by_key(grouped_values, reducer):
    return [reducer(key, values) for key, values in grouped_values.items()]

In [7]:
# Este reducer del ejemplo suma las horas de cada conductor.

def sum_driver_hours(driver_id, hours):
    return driver_id, sum(hours)


total_hour_pairs = reduce_by_key(hours_by_driver, sum_driver_hours)

total_hour_pairs

[('10', 140.0), ('11', 133.0)]

In [8]:
# El mismo operador de mapeo prepara una segunda fuente con la misma clave.

def map_driver_name(record):
    return [(record["driverId"], record["name"])]


driver_name_pairs = map_pairs(sample_drivers, map_driver_name)

driver_name_pairs

[('10', 'George Vetticaden'), ('11', 'Jamie Engesser')]

In [9]:
# inner_join_by_key recibe dos colecciones de parejas y una regla para combinar sus valores.

def inner_join_by_key(left_pairs, right_pairs, joiner):
    right_values = dict(right_pairs)

    return [
        joiner(key, left_value, right_values[key])
        for key, left_value in left_pairs
        if key in right_values
    ]

In [10]:
# Operadores complementarios: conservan la misma idea de transformación sobre colecciones.

def filter_records(records, predicate):
    return [record for record in records if predicate(record)]


def map_values(pairs, mapper):
    return [(key, mapper(value)) for key, value in pairs]


def flat_map_pairs(records, mapper):
    pairs = []

    for record in records:
        pairs.extend(mapper(record))

    return pairs


def union_pairs(left_pairs, right_pairs):
    return [*left_pairs, *right_pairs]


def sort_by_key(pairs):
    return sorted(pairs, key=lambda pair: pair[0])


def left_outer_join_by_key(left_pairs, right_pairs, joiner):
    right_values = group_by_key(right_pairs)

    return [
        joiner(key, left_value, right_values.get(key, []))
        for key, left_value in left_pairs
    ]


filtered_pairs = map_pairs(
    filter_records(sample_timesheet, lambda record: record['week'] == '1'),
    map_timesheet_hours,
)
normalized_pairs = map_values(filtered_pairs, lambda hours: hours / 10)
sorted_pairs = sort_by_key(union_pairs(normalized_pairs, []))

sorted_pairs

[('10', 7.0), ('11', 5.0)]

In [11]:
# La regla del ejemplo decide cómo presentar el nombre y las horas agregadas.

def combine_driver_hours(driver_id, total_hours, name):
    return {"driverId": driver_id, "name": name, "total_hours": total_hours}


operator_walkthrough = inner_join_by_key(
    total_hour_pairs,
    driver_name_pairs,
    combine_driver_hours,
)

with (ROOT / "submission" / "operator_walkthrough.csv").open(
    "w", encoding="utf-8", newline=""
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["driverId", "name", "total_hours"],
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(operator_walkthrough)

operator_walkthrough

[{'driverId': '10', 'name': 'George Vetticaden', 'total_hours': 140.0},
 {'driverId': '11', 'name': 'Jamie Engesser', 'total_hours': 133.0}]